# 03 — Modelo Preditivo: Detecção de Lotes Fora do Padrão

Neste notebook construímos um modelo de classificação para prever se um lote
será **APROVADO** ou **REPROVADO** com base nos parâmetros de processo e qualidade.

**Etapas:**
1. Pré-processamento
2. Baseline — Regressão Logística
3. Modelo principal — Random Forest
4. Avaliação de desempenho
5. Interpretabilidade — Feature Importance
6. Conclusões

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/raw/lotes_qc.csv', parse_dates=['data_producao'])
print(f"Shape: {df.shape}")
df.head()

## 1. Pré-processamento

In [ ]:
# Encoding da variável categórica 'turno'
le_turno = LabelEncoder()
df['turno_enc'] = le_turno.fit_transform(df['turno'])

# Variável alvo
df['target'] = (df['status_lote'] == 'REPROVADO').astype(int)

# Features e target
FEATURES = ['temp_processo', 'umidade_relativa', 'pressao_compressao',
            'dureza_media', 'friabilidade', 'peso_medio', 'dissolucao', 'turno_enc']

X = df[FEATURES]
y = df['target']

# Split treino/teste (80/20, estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalização
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Proporção de reprovados — Treino: {y_train.mean():.1%} | Teste: {y_test.mean():.1%}")

## 2. Baseline — Regressão Logística

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)

y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

print("=== Regressão Logística (Baseline) ===")
print(classification_report(y_test, y_pred_lr, target_names=['APROVADO', 'REPROVADO']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

## 3. Modelo Principal — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['APROVADO', 'REPROVADO']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

# Cross-validation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')
print(f"\nCross-validation ROC-AUC (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 4. Avaliação de Desempenho

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusão
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['APROVADO', 'REPROVADO']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — Random Forest', fontweight='bold')

# Curva ROC — comparação
for nome, y_prob, cor in [
    ('Regressão Logística', y_prob_lr, '#3498db'),
    ('Random Forest',       y_prob_rf, '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, label=f'{nome} (AUC={auc:.3f})', color=cor, linewidth=2)

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_title('Curva ROC — Comparação de Modelos', fontweight='bold')
axes[1].set_xlabel('Taxa de Falso Positivo')
axes[1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/06_avaliacao_modelos.png')
plt.show()

## 5. Interpretabilidade — Feature Importance

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
cores = ['#e74c3c' if v >= importancias.quantile(0.75) else '#3498db' for v in importancias]
importancias.plot(kind='barh', ax=ax, color=cores, edgecolor='white')
ax.set_title('Importância das Variáveis — Random Forest', fontweight='bold')
ax.set_xlabel('Importância Relativa')
ax.axvline(importancias.mean(), color='gray', linestyle='--', label='Média')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/07_feature_importance.png')
plt.show()

print("\nVariáveis mais importantes:")
print(importancias.sort_values(ascending=False).round(4))

## 6. Conclusões

### Desempenho dos Modelos
| Modelo | ROC-AUC |
|---|---|
| Regressão Logística (baseline) | *ver saída acima* |
| Random Forest | *ver saída acima* |

### Principais Achados
- As variáveis com maior poder preditivo foram **friabilidade** e **dissolução** — parâmetros críticos de qualidade diretamente relacionados às especificações do produto
- **Pressão de compressão** se destacou entre as variáveis de processo, confirmando sua influência sobre os atributos de qualidade identificados na EDA
- O modelo Random Forest superou o baseline de Regressão Logística, especialmente no recall para lotes REPROVADOS — comportamento desejável em ambiente regulado, onde falsos negativos têm alto custo

### Aplicação Prática
Um modelo como este, integrado a um sistema MES ou LIMS, poderia:
- Sinalizar lotes com alto risco de reprovação ainda durante o processo produtivo
- Direcionar inspeções de qualidade de forma prioritária
- Reduzir retrabalho e desperdício de insumos farmacêuticos

### Próximos Passos
- Explorar ajuste de hiperparâmetros (GridSearchCV)
- Testar XGBoost e comparar desempenho
- Adicionar análise SHAP para interpretabilidade avançada